In [ ]:
import os
import gc
import torch
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# ---------------------------------------------------------
# [긴급] 메모리 청소 및 환경 설정
# ---------------------------------------------------------
# 1. 토크나이저 병렬처리 끄기 (Deadlock 방지)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. 캐시 비우기
gc.collect()
torch.cuda.empty_cache()

# 3. 시드 고정
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
print("🚀 [Step 1] 안전 모드(Safe Mode) 학습 시작")
# ---------------------------------------------------------

# 2. 데이터 로드 및 전처리
DATA_PATH = "./data/munch_labeled_RESUMED_200125.csv" 
df = pd.read_csv(DATA_PATH)

df_train = df[['original', 'score']].copy()
df_train.rename(columns={'original': 'text', 'score': 'label'}, inplace=True)
df_train.dropna(subset=['text', 'label'], inplace=True)

# [Label Normalization] 1~5점 -> 0.0~1.0 (학습 안정성 확보)
df_train['label'] = (df_train['label'].astype(float) - 1) / 4.0

dataset = Dataset.from_pandas(df_train)
train_test_split = dataset.train_test_split(test_size=0.15, seed=SEED)
dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

# 3. 모델 로드 (SOTA: RoBERTa)
MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=1, 
    problem_type="regression"
)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. 학습 파라미터 (메모리 안전 설정)
OUTPUT_DIR = "./output/smart_router_safe"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,             
    learning_rate=3e-5,
    
    # [핵심 변경] 배치 사이즈 16으로 축소 (메모리 터짐 방지)
    per_device_train_batch_size=16,
    
    # [핵심 기술] Gradient Accumulation (배치 16을 2번 모아서 32 효과 냄)
    gradient_accumulation_steps=2,
    
    per_device_eval_batch_size=64,   # 평가는 메모리 적게 드므로 유지
    
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",      # 부드러운 학습률 조정
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_rmse",
    greater_is_better=False,
    
    report_to="none",
    seed=SEED,
    fp16=True,                       # 속도 향상 & 메모리 절약
    
    # [핵심 변경] 워커 수 0 (프로세스 충돌 방지)
    dataloader_num_workers=0,        
    group_by_length=True,
)

# 평가 함수 (Unscaling 포함)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.squeeze()
    
    # 0~1로 압축된 값을 다시 1~5로 복원
    pred_restored = np.clip(predictions, 0, 1) * 4.0 + 1.0
    label_restored = labels * 4.0 + 1.0
    
    mse = mean_squared_error(label_restored, pred_restored)
    rmse = np.sqrt(mse)
    r2 = r2_score(label_restored, pred_restored)
    
    return {"mse": mse, "rmse": rmse, "r2": r2}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# 5. 학습 실행
print("\n🔥 [Safe Mode] 학습 시작!")
trainer.train()

# 6. 최적 임계값 탐색 (변수명 수정됨)
print("\n🔍 최적의 라우팅 임계값 탐색 중...")
predictions = trainer.predict(tokenized_datasets["test"]).predictions.squeeze()

# [수정된 부분] 변수명을 true_restored로 통일했습니다.
pred_restored = np.clip(predictions, 0, 1) * 4.0 + 1.0
true_restored = np.array(tokenized_datasets["test"]["label"]) * 4.0 + 1.0 

# 실제 정답(4점 이상 Hard) 기준 생성
true_binary = [1 if x >= 4.0 else 0 for x in true_restored]

best_threshold = 3.5
best_acc = 0.0

# 2.0 ~ 4.5 범위 탐색
for th in np.arange(2.0, 4.6, 0.1):
    pred_binary = [1 if x >= th else 0 for x in pred_restored]
    acc = accuracy_score(true_binary, pred_binary)
    if acc > best_acc:
        best_acc = acc
        best_threshold = th

print(f"\n🏆 [최종 결과]")
print(f"  - RMSE (낮을수록 좋음): {trainer.evaluate()['eval_rmse']:.4f}")
print(f"  - 최적 임계값(Threshold): {best_threshold:.1f}")
print(f"  - 최적화된 정확도(Accuracy): {best_acc:.4f}")

# 7. 저장
trainer.save_model(OUTPUT_DIR)
with open(os.path.join(OUTPUT_DIR, "best_threshold.txt"), "w") as f:
    f.write(str(best_threshold))

print(f"\n💾 저장 완료: {OUTPUT_DIR}")

1. 결과 분석 및 평가 (Post-Mortem)
"안정성은 확보했지만, 성능(정확도)은 동전 던지기(52%) 수준으로 떨어졌습니다."

원인 분석:

회귀(Regression)의 실패: 문장의 난이도 점수(1~5점)가 수학적으로 선형적인 관계가 아닐 가능성이 큽니다. (예: 3점과 4점의 차이가 1점과 2점의 차이와 의미적으로 다름).

데이터의 모호성: Gemini가 매긴 점수 자체가 미세한 차이를 반영하기엔 노이즈가 있을 수 있습니다.

모델의 혼란: 모델이 정확한 점수를 맞추려다 보니, 오히려 0(Easy)과 1(Hard)을 가르는 결정적인 특징을 놓치고 "안전한 평균값"만 내뱉게 되었습니다.